In [14]:
#!/usr/bin/env python
# coding: utf-8

# 导入标准库：用于文件/路径操作
import os
# 导入项目内的日志/显示工具，便于打印信息与进度提示
import hues
# 用于将 Python 对象序列化到磁盘（保存 scaler、meta 等）
import pickle
# 数值计算核心库，主要用于数组/矩阵运算
import numpy as np
# 数据读取与 DataFrame 操作（CSV/PKL 等）
import pandas as pd

# 从 tools 模块导入自定义工具：CustomScaler 用于拟合/变换数据，check_folder 用于确保保存目录存在
%cd D:\大学\博士论文\GNN部分\07_RA-STGNN
from tools import CustomScaler, check_folder

D:\大学\博士论文\GNN部分\07_RA-STGNN


C:\Users\MR.long\AppData\Roaming\Python\Python39\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [15]:
# =========================
# 0) 可调参数（先按推荐默认值）

# 输入历史窗口长度（模型输入的时间步数）
WINDOW_SIZE = 48  # 输入历史窗口长度
# 预测窗口长度（模型要预测的未来步数）
HORIZON_SIZE = 12  # 预测窗口长度

# 训练/验证划分比例：在 合并 的数据上进一步划分为 train/val
TRAIN_RATIO = 0.7

# 下采样间隔（秒），None 表示不下采样；数值例如 10 表示每 10 秒采样一次
DOWNSAMPLE_SEC = 10  # 推荐 10；如要保持 1s：改成 None

# 保存结果的路径，包含窗口与预测长度等信息，便于区分不同配置的输出文件夹
save_path = f"D:/大学/博士论文/GNN部分/07_RA-STGNN/5_AnomalyDetection/SWAT_STGNN_[{WINDOW_SIZE}_{HORIZON_SIZE}_{DOWNSAMPLE_SEC}]/"

In [16]:
# 从磁盘读取预先保存的 DataFrame（pickle 格式），路径为相对路径到项目的 Data 文件夹
df = pd.read_pickle('D:/大学/博士论文/GNN部分/07_RA-STGNN/Data/SWaT/swat_data_10s.pkl')

# 使用 hues 工具打印 DataFrame 的形状（行, 列），帮助确认数据读取是否正确
hues.info(df.shape)

# 显示 DataFrame 的前几行，便于快速检查列名与部分数据样例
df.head()

01:08:15 - INFO - (94673, 54)


,Timestamp,Attack,Train,FIT101,LIT101,MV101,P101,P102,AIT201,AIT202,...,FIT504,P501,P502,PIT501,PIT502,PIT503,FIT601,P601,P602,P603
0,2015-12-22 16:00:00,0,1,2.479006,260.68542,2.0,2.0,1.0,244.48859,8.190080,...,0.0,1.0,1.0,10.029480,0.0,4.277749,0.000256,1.0,1.0,1.0
1,2015-12-22 16:00:10,0,1,2.619607,261.36449,2.0,2.0,1.0,244.95640,8.190080,...,0.0,1.0,1.0,10.029480,0.0,4.277749,0.000256,1.0,1.0,1.0
2,2015-12-22 16:00:20,0,1,2.482433,262.00037,2.0,2.0,1.0,245.44990,8.192226,...,0.0,1.0,1.0,10.048706,0.0,4.277749,0.000256,1.0,1.0,1.0
3,2015-12-22 16:00:30,0,1,2.573135,262.40076,2.0,2.0,1.0,245.68057,8.193604,...,0.0,1.0,1.0,10.056717,0.0,4.277749,0.000256,1.0,1.0,1.0
4,2015-12-22 16:00:40,0,1,2.571117,262.18488,2.0,2.0,1.0,246.04270,8.196168,...,0.0,1.0,1.0,10.029480,0.0,4.277749,0.000256,1.0,1.0,1.0


In [17]:
# 缺失值检查：计算整个 DataFrame 中缺失值（NaN）的总数
df.isna().sum().sum()

np.int64(0)

In [18]:
# =========================
# 3) 列类型划分：特征列 / 标签列
# =========================
# 指定标签列（时间戳、Train 标记、Attack 标记）
label_cols = ["Timestamp", "Train", "Attack"]

# 特征列：除了标签列之外的所有列都作为特征列使用
feature_cols = [c for c in df.columns if c not in label_cols]

# =========================
# 状态/开关列识别逻辑：通过检查前10行判断是否为离散数据
# 离散数据特征：值为小整数（如 0, 1, 2, 3 等）的有限集合
# =========================
def is_discrete_column(col_data, check_rows=10):
    """
    判断一列是否为离散数据列
    参数：
        col_data: pandas Series 列数据
        check_rows: 检查前N行来判断是否为离散数据
    返回：
        True 表示该列为离散数据，False 表示连续数据
    """
    # 取前 check_rows 行的非空值
    sample = col_data.dropna().head(check_rows)
    
    # 如果没有足够的样本数据，默认不判定为离散
    if len(sample) == 0:
        return False
    
    try:
        # 尝试将样本值转换为浮点数再转整数
        # 检查转换后是否与原值相等（即判断是否为整数）
        sample_float = sample.astype(float)
        sample_int = sample_float.astype(int)
        
        # 判断所有值是否都是整数（浮点与整数表示相同）
        is_all_integer = (sample_float == sample_int).all()
        
        # 获取样本中的唯一值个数
        unique_count = len(sample.unique())
        
        # 判断唯一值是否较少（通常离散数据的唯一值个数 <= 10）
        is_low_cardinality = unique_count <= 10
        
        # 综合判断：既然都是整数，且唯一值较少，则认定为离散数据
        return is_all_integer and is_low_cardinality
    except:
        # 如果转换失败，说明不是整数数据，不认定为离散数据
        return False

# 遍历所有特征列，识别哪些是离散列
status_cols = []
for c in feature_cols:
    if is_discrete_column(df[c], check_rows=10):
        status_cols.append(c)

# 连续变量列：其余都按连续量处理（PV、流量、压力等）
cont_cols = [c for c in feature_cols if c not in status_cols]

# 使用 hues 打印列的统计信息，便于确认列分类是否合理
hues.info(f"Feature cols [{len(feature_cols)}]: ", str(feature_cols))
hues.info(f"Discrete (Status) cols [{len(status_cols)}]: ", str(status_cols))
hues.info(f"Continuous cols [{len(cont_cols)}]: ", str(cont_cols))

01:08:22 - INFO - Feature cols [51]:  ['FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201', 'P201', 'P202', 'P203', 'P204', 'P205', 'P206', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402', 'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'P502', 'PIT501', 'PIT502', 'PIT503', 'FIT601', 'P601', 'P602', 'P603']
01:08:22 - INFO - Discrete (Status) cols [30]:  ['MV101', 'P101', 'P102', 'MV201', 'P201', 'P202', 'P203', 'P204', 'P205', 'P206', 'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302', 'AIT401', 'FIT401', 'P401', 'P402', 'P403', 'P404', 'UV401', 'FIT504', 'P501', 'P502', 'PIT502', 'P601', 'P602', 'P603']
01:08:22 - INFO - Continuous cols [21]:  ['FIT101', 'LIT101', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'DPIT301', 'FIT301', 'LIT301', 'AIT402', 'LIT401', 'AIT501', 'AIT502', 'AIT503', 'AIT504'

In [19]:
# =========================
# 验证离散列的识别结果：查看前10行样本和数据分布
# =========================

# 如果识别出离散列，则详细检查和展示结果
if len(status_cols) > 0:
    print("=" * 70)
    print("DISCRETE COLUMNS - First 10 rows sample:")
    print("=" * 70)
    # 显示所有离散列的前10行数据，便于目视验证
    print(df[status_cols].head(10))
    
    print("\n" + "=" * 70)
    print("DISCRETE COLUMNS - Data info (dtype, unique values, value counts):")
    print("=" * 70)
    # 逐个离散列进行详细分析
    for c in status_cols:
        # 获取该列的数据类型
        col_dtype = df[c].dtype
        # 获取非空值并计算唯一值
        non_null_values = df[c].dropna()
        unique_vals = sorted(non_null_values.unique())
        
        print(f"\n[{c}]")
        print(f"  Data type: {col_dtype}")
        print(f"  Unique values ({len(unique_vals)}): {unique_vals}")
        
        # 显示该列的值分布（频数统计）
        vc = df[c].value_counts(dropna=False).sort_index()
        print(f"  Value counts:")
        for val, count in vc.items():
            print(f"    {val}: {count}")

# 如果识别出连续列，也显示其样本以便对比
if len(cont_cols) > 0:
    print("\n" + "=" * 70)
    print("CONTINUOUS COLUMNS - First 10 rows sample:")
    print("=" * 70)
    # 显示所有连续列的前10行数据
    print(df[cont_cols].head(10))
    
    print("\n" + "=" * 70)
    print("CONTINUOUS COLUMNS - Basic statistics:")
    print("=" * 70)
    # 显示连续列的基本统计信息（均值、标准差、最小值、最大值等）
    print(df[cont_cols].describe())

DISCRETE COLUMNS - First 10 rows sample:
   MV101  P101  P102  MV201  P201  P202  P203  P204  P205  P206  ...  P403  \
0    2.0   2.0   1.0    2.0   1.0   1.0   2.0   1.0   2.0   1.0  ...   1.0   
1    2.0   2.0   1.0    2.0   1.0   1.0   2.0   1.0   2.0   1.0  ...   1.0   
2    2.0   2.0   1.0    2.0   1.0   1.0   2.0   1.0   2.0   1.0  ...   1.0   
3    2.0   2.0   1.0    2.0   1.0   1.0   2.0   1.0   2.0   1.0  ...   1.0   
4    2.0   2.0   1.0    2.0   1.0   1.0   2.0   1.0   2.0   1.0  ...   1.0   
5    2.0   2.0   1.0    2.0   1.0   1.0   2.0   1.0   2.0   1.0  ...   1.0   
6    2.0   2.0   1.0    2.0   1.0   1.0   2.0   1.0   2.0   1.0  ...   1.0   
7    2.0   2.0   1.0    2.0   1.0   1.0   2.0   1.0   2.0   1.0  ...   1.0   
8    2.0   2.0   1.0    2.0   1.0   1.0   2.0   1.0   2.0   1.0  ...   1.0   
9    2.0   2.0   1.0    2.0   1.0   1.0   2.0   1.0   2.0   1.0  ...   1.0   

   P404  UV401  FIT504  P501  P502  PIT502  P601  P602  P603  
0   1.0    1.0     0.0   1.0   1.0   

In [20]:
# 在这里的划分时，对于IS-BLS算法必须是合并的数据按7:1:2 划分为 Train:Val:Test
# 对于ASDP算法则是先切分 Train==1 / Train==0，Train==1 用于训练与验证，Train==0 单独作为测试集

# =========================
# 4) 切分 Train==1 / Train==0
# =========================
# 对于ASDP算法将数据按列 'Train' 的值分开：Train==1 用于训练与验证，Train==0 单独作为测试集
df_train_val = df[df["Train"] == 1].copy()
df_test = df[df["Train"] == 0].copy()

# 打印切分后的原始形状（未下采样前）
hues.info(("TrainVal raw:", df_train_val.shape))
hues.info(("Test raw:", df_test.shape))

# 为减少后续计算，仅保留必要的列（仍保留 Timestamp/Train/Attack 便于下采样时对齐）
df_train_val = df_train_val[["Timestamp", "Train", "Attack"] + feature_cols]
df_test = df_test[["Timestamp", "Train", "Attack"] + feature_cols]

01:08:34 - INFO - ('TrainVal raw:', (49680, 54))
01:08:34 - INFO - ('Test raw:', (44993, 54))


In [21]:
# 导入下采样函数：将连续时间序列按照指定秒数聚合/采样（实际并没有设置参数）
from tools import downsample_block

# 如果指定了 DOWNSAMPLE_SEC，则对 train/val/test 分别进行下采样处理
if DOWNSAMPLE_SEC is not None:
    # downsample_block 函数会在内部对连续与状态列分别聚合，保持时间对齐
    df_train_val = downsample_block(df_train_val, cont_cols, status_cols, DOWNSAMPLE_SEC)
    df_test = downsample_block(df_test, cont_cols, status_cols, DOWNSAMPLE_SEC)

# 打印下采样后的数据形状，便于验证是否按预期减少了时间步数
hues.info(("TrainVal after downsample:", df_train_val.shape))
hues.info(("Test after downsample:", df_test.shape))

01:08:38 - INFO - ('TrainVal after downsample:', (49680, 54))
01:08:38 - INFO - ('Test after downsample:', (44993, 54))


In [22]:
# =========================
# 6) 训练/验证按时间顺序切分（在 Train==1 内部再切）
#    说明：
#    - 由于测试集由 Train==0 单独给出，我们不再保留“train/val/test=0.7/0.2/0.1”的第三段
#    - 这里把 Train==1 的数据全部用完：train_ratio = 0.7/(0.7+0.2), val_ratio = 0.2/(0.7+0.2)
# =========================
# 计算 Train==1 部分的总长度（时间序列条数）
tv_len = len(df_train_val)

# 根据 TRAIN_RATIO 计算切分索引（前段作为 train，后段作为 val）
split_idx = int(tv_len * TRAIN_RATIO)

# 按时间顺序切分：前 split_idx 行作为训练集，剩余作为验证集
df_train = df_train_val.iloc[:split_idx].copy()
df_val = df_train_val.iloc[split_idx:].copy()

# 打印各分集形状，方便检查
hues.info("df_train:", df_train.shape)
hues.info("df_val:", df_val.shape)
hues.info("df_test:", df_test.shape)

01:08:41 - INFO - df_train: (34776, 54)
01:08:41 - INFO - df_val: (14904, 54)
01:08:41 - INFO - df_test: (44993, 54)


In [ ]:
# =========================
# 7) 构造 归一化scaler（只用 train 拟合，避免数据泄露）
#    连续列：z-score（按列）
#    状态列：不归一化（把 mean=0,std=1，相当于 transform 后原值不变）
# =========================
# 从训练集提取特征值并转换为 float64，供 scaler 拟合使用
train_values = df_train[feature_cols].values.astype(np.float64)

# 使用 CustomScaler 在训练数据上拟合（不使用 one-hot 编码）
scaler = CustomScaler(train_values, use_one=False)

# 对状态列做特殊处理：人为设置状态列的 mean=0, std=1，使得 transform 后状态列保持原始取值（不归一化）
if len(status_cols) > 0:
    # 找到状态列在 feature_cols 中的索引位置
    status_idx = [feature_cols.index(c) for c in status_cols]
    # 将这些索引对应的均值设置为 0
    scaler.mean[status_idx] = 0.0
    # 将这些索引对应的标准差设置为 1
    scaler.std[status_idx] = 1.0

# 打印提示，说明 scaler 已在训练数据上拟合完成
hues.success("Scaler fitted on df_train. Status cols are kept as-is (mean=0,std=1).")

请注意以下列的标准差为零:
列 [4] 标准差为零, 所有值相同.
列 [10] 标准差为零, 所有值相同.
列 [11] 标准差为零, 所有值相同.
列 [13] 标准差为零, 所有值相同.
列 [15] 标准差为零, 所有值相同.
列 [29] 标准差为零, 所有值相同.
列 [31] 标准差为零, 所有值相同.
列 [32] 标准差为零, 所有值相同.
列 [43] 标准差为零, 所有值相同.
列 [48] 标准差为零, 所有值相同.
列 [50] 标准差为零, 所有值相同.
01:08:47 - SUCCESS - Scaler fitted on df_train. Status cols are kept as-is (mean=0,std=1).


In [24]:
# =========================
# 8) 标准化并生成滑动窗口样本 (x,y)
#    x: [num_samples, WINDOW_SIZE, num_nodes, 1]
#    y: [num_samples, HORIZON_SIZE, num_nodes, 1]
# =========================
def build_xy_from_df(_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    # 只取特征列（按之前的 feature_cols 顺序）
    data_raw = _df[feature_cols].values.astype(np.float64)

    # 在训练好的 scaler 上进行变换，得到标准化后的数据（float32 以节省内存）
    data_std = scaler.transform(data_raw).astype(np.float32)

    # 增加最后一维，表示特征维度（此处假设每节点只有 1 个特征），结果形状为 [T, N, 1]
    data_std = np.expand_dims(data_std, axis=-1)  # [T, N, 1]

    # 解包时间步长 T、节点数 N、特征数 F（此处 F 应为 1）
    T, N, F = data_std.shape

    # 计算可生成的样本数：从 t=0 到 t=T-(WINDOW+HORIZON)
    num_samples = T - (WINDOW_SIZE + HORIZON_SIZE) + 1
    if num_samples <= 0:
        # 如果时间步不够，抛出异常提醒用户调整窗口/预测长度或使用更多数据
        raise ValueError(f"Not enough timesteps: T={T}, WINDOW={WINDOW_SIZE}, HORIZON={HORIZON_SIZE}")

    # 为 x 和 y 申请数组，类型为 float32，形状按照模型输入输出要求构造
    x = np.zeros((num_samples, WINDOW_SIZE, N, 1), dtype=np.float32)
    y = np.zeros((num_samples, HORIZON_SIZE, N, 1), dtype=np.float32)

    # 逐样本填充：每个样本的 x 是连续的 WINDOW_SIZE 步，y 是紧随其后的 HORIZON_SIZE 步
    for i in range(num_samples):
        # t0 为当前样本的起始时间索引
        t0 = i
        # 取从 t0 到 t0+WINDOW_SIZE-1 的数据作为输入 x
        x[i] = data_std[t0: t0 + WINDOW_SIZE]
        # 取紧随其后的 HORIZON_SIZE 步作为标签 y
        y[i] = data_std[t0 + WINDOW_SIZE: t0 + WINDOW_SIZE + HORIZON_SIZE]

    return x, y


# 基于不同分集构建样本：train/val/test
x_train, y_train = build_xy_from_df(df_train)
x_val, y_val = build_xy_from_df(df_val)
x_test, y_test = build_xy_from_df(df_test)

# 打印各分集构建后 x/y 的形状信息，便于检查样本维度是否符合模型预期
hues.info("train x:", x_train.shape, "y:", y_train.shape)
hues.info("val   x:", x_val.shape, "y:", y_val.shape)
hues.info("test  x:", x_test.shape, "y:", y_test.shape)

01:08:53 - INFO - train x: (34717, 48, 51, 1) y: (34717, 12, 51, 1)
01:08:53 - INFO - val   x: (14845, 48, 51, 1) y: (14845, 12, 51, 1)
01:08:53 - INFO - test  x: (44934, 48, 51, 1) y: (44934, 12, 51, 1)


In [25]:
# 确保保存目录存在；如果不存在则创建（check_folder 内部实现）
check_folder(save_path)

# 保存 Numpy 压缩文件（.npz），包含 x 与 y，分别保存 train/val/test
for cat in ["train", "val", "test"]:
    # 使用 locals() 获取对应变量 x_train/x_val/x_test 与 y_*
    _x = locals()[f"x_{cat}"]
    _y = locals()[f"y_{cat}"]

    # 使用 np.savez_compressed 以节省磁盘空间，将 x/y 保存到单个压缩文件中
    np.savez_compressed(
        os.path.join(save_path, f"{cat}.npz"),
        x=_x.astype(np.float32),
        y=_y.astype(np.float32),
    )
    # 打印保存成功的信息以及文件中 x/y 的形状
    hues.success(f"Saved {cat}.npz -> x:{_x.shape}, y:{_y.shape}")

# 保存 scaler（用于后续推理时对数据做相同的标准化），使用 pickle 序列化
with open(os.path.join(save_path, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)
hues.success(f"Saved scaler.pkl -> {save_path}")

01:09:02 - INFO - Successfully Created the folder: ['D:/大学/博士论文/GNN部分/07_RA-STGNN/5_AnomalyDetection/SWAT_STGNN_[48_12_10]/'].
01:09:04 - SUCCESS - Saved train.npz -> x:(34717, 48, 51, 1), y:(34717, 12, 51, 1)
01:09:04 - SUCCESS - Saved val.npz -> x:(14845, 48, 51, 1), y:(14845, 12, 51, 1)
01:09:07 - SUCCESS - Saved test.npz -> x:(44934, 48, 51, 1), y:(44934, 12, 51, 1)
01:09:07 - SUCCESS - Saved scaler.pkl -> D:/大学/博士论文/GNN部分/07_RA-STGNN/5_AnomalyDetection/SWAT_STGNN_[48_12_10]/


In [26]:
# =========================
# 10) （可选）保存一些元信息，方便你以后对齐列顺序/类型
# =========================
# 将配置与列顺序等元信息保存到一个 dict 中，便于模型加载时恢复一致的列顺序
meta = {
    "window_size": WINDOW_SIZE,
    "horizon_size": HORIZON_SIZE,
    "downsample_sec": DOWNSAMPLE_SEC,
    "feature_cols": feature_cols,
    "status_cols": status_cols,
    "cont_cols": cont_cols,
    "train_ratio_eff": TRAIN_RATIO
}
with open(os.path.join(save_path, "meta.pkl"), "wb") as f:
    pickle.dump(meta, f)
hues.success("Saved meta.pkl (feature order & config).")

01:09:48 - SUCCESS - Saved meta.pkl (feature order & config).
